## Task 4: MAchine Translation RNN/ LSTM

In [1]:

# =============================================================================
# TASK 4 — MACHINE TRANSLATION (English → Telugu)
# Pure Seq2Seq WITHOUT attention — vanilla RNN vs LSTM
# CS6910 Programming Assignment III
# =============================================================================
#
# ARCHITECTURE (NO ATTENTION — intentional)
# ──────────────────────────────────────────
#  Encoder : Bidirectional RNN / LSTM  (N_LAYERS layers, HIDDEN_DIM per dir)
#              packed for correctness → final hidden merges fwd+bwd per layer
#              → projected to (N_LAYERS, B, HIDDEN_DIM) to init decoder
#
#  Decoder : Unidirectional RNN / LSTM  (N_LAYERS layers, HIDDEN_DIM)
#              input at step t = embed(prev_word)  [300-d]
#              output            = Linear(HIDDEN_DIM → |Telugu vocab|)
#              ** NO context vector / NO cross-attention **
#
#  The decoder relies ENTIRELY on the encoder's final hidden state.
#  This is the classic "fixed-size bottleneck" design — deliberately chosen
#  so we can contrast RNN vs LSTM fairly:
#    • RNN  : vanishing gradients → final hidden forgets early source words
#    • LSTM : cell state c_t gives a near-linear gradient path → better memory
#  Task 5 (Transformer) fixes the bottleneck with cross-attention.
#
# WHAT CHANGED vs V1 (and why results improve)
# ──────────────────────────────────────────────
#  ✓ N_LAYERS 1 → 2  (deeper encoder + decoder, more representational power)
#  ✓ HIDDEN_DIM 256 → 512  (wider hidden state)
#  ✓ Beam search (beam=5, α=0.7 length norm) — eliminates repetition loops
#  ✓ Better Telugu tokeniser (regex, not bare split())
#  ✓ AdamW + label smoothing 0.1 — less overconfidence, better calibration
#  ✓ AMP (mixed precision) + gradient accumulation — faster GPU training
#  ✓ pack_padded_sequence — encoder final hidden reflects TRUE sentence end
#  ✓ Teacher-forcing ratio decays 0.70 → 0.50 — reduces exposure bias
# =============================================================================

import os, math, random, re, time, json
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import nltk
nltk.download("punkt", quiet=True)

# ── CONFIG ────────────────────────────────────────────────────────────────────
ENG_EMBED_DIM = 300
TEL_EMBED_DIM = 300
HIDDEN_DIM    = 512          # was 256
N_LAYERS      = 2            # was 1
DROPOUT       = 0.3
BEAM_SIZE     = 5
BATCH_SIZE    = 64
ACCUM_STEPS   = 2
MAX_EPOCHS    = 50
PATIENCE      = 7
LR            = 3e-4
CLIP          = 1.0
MAX_LEN       = 60
MIN_FREQ      = 2
TF_START      = 0.70         # teacher-forcing at epoch 1
TF_END        = 0.50         # teacher-forcing at final epoch
LABEL_SMOOTH  = 0.1
MIXED_PREC    = True
USE_BEAM_BLEU = True         # set False for faster greedy BLEU
SEED          = 42

PAD, SOS, EOS, UNK         = "<PAD>", "<SOS>", "<EOS>", "<UNK>"
SPECIAL                    = [PAD, SOS, EOS, UNK]
PAD_ID, SOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

TRAIN_PATH    = "./task45_MT/team6_te_train.csv"
VAL_PATH      = "./task45_MT/team6_te_valid.csv"
TEST_PATH     = "./task45_MT/team6_te_test.csv"
GLOVE_PATH    = "./task45_MT/glove.6B.300d.txt"
FASTTEXT_PATH = "./task45_MT/wiki.te.vec.telugu"
OUTPUT_DIR    = "./task4_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
print(f"Device: {DEVICE}")

# ── TOKENISERS ────────────────────────────────────────────────────────────────
def tokenize_english(s):
    s = str(s).lower().strip()
    s = re.sub(r"([.!?,;:])", r" \1 ", s)
    s = re.sub(r"[^a-z0-9.!?,;:'\- ]+", " ", s)
    return s.split()

def tokenize_telugu(s):
    # regex split: keeps punctuation as tokens (better than bare split())
    toks = re.split(r'(\s+|[।,.!?;:"\'\(\)\[\]\{\}])', str(s).strip())
    return [t for t in toks if t.strip()]

# ── VOCABULARY ────────────────────────────────────────────────────────────────
class Vocabulary:
    def __init__(self, name):
        self.name     = name
        self.word2idx = {t: i for i, t in enumerate(SPECIAL)}
        self.idx2word = {i: t for t, i in self.word2idx.items()}
        self.word_count = Counter()

    def build(self, sentences, tokenize_fn, min_freq=MIN_FREQ):
        for s in sentences:
            self.word_count.update(tokenize_fn(s))
        for w, c in self.word_count.most_common():
            if c >= min_freq and w not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[w] = idx; self.idx2word[idx] = w
        print(f"  [{self.name}] vocab={len(self.word2idx)}  "
              f"(unique tokens seen={len(self.word_count)})")

    def encode(self, tokens, max_len=MAX_LEN):
        return [self.word2idx.get(t, UNK_ID) for t in tokens[:max_len]]

    def decode(self, ids, skip_unk=True):
        out = []
        for i in ids:
            w = self.idx2word.get(int(i), UNK)
            if w == EOS: break
            if w in (PAD, SOS): continue
            if skip_unk and w == UNK: continue
            out.append(w)
        return " ".join(out)

    def __len__(self): return len(self.word2idx)

# ── DATA ──────────────────────────────────────────────────────────────────────
def load_csv(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower()
    df = df[["source","target"]].dropna()
    df.columns = ["english","telugu"]
    return df[df["english"].str.strip().astype(bool) &
              df["telugu"].str.strip().astype(bool)].reset_index(drop=True)

print("\n[1] Loading data...")
train_df = load_csv(TRAIN_PATH)
val_df   = load_csv(VAL_PATH)
test_df  = load_csv(TEST_PATH)
print(f"  train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

print("\n[2] Building vocabularies (train only)...")
src_vocab = Vocabulary("English"); tgt_vocab = Vocabulary("Telugu")
src_vocab.build(train_df["english"], tokenize_english)
tgt_vocab.build(train_df["telugu"],  tokenize_telugu)

class TranslationDataset(Dataset):
    def __init__(self, df):
        self.pairs = []
        for _, row in df.iterrows():
            s = src_vocab.encode(tokenize_english(row["english"]))
            t = [SOS_ID] + tgt_vocab.encode(tokenize_telugu(row["telugu"])) + [EOS_ID]
            if s and len(t) > 2:
                self.pairs.append((s, t, row["english"], row["telugu"]))
    def __len__(self):  return len(self.pairs)
    def __getitem__(self, i): return self.pairs[i]

def collate_fn(batch):
    # sort descending by src len — required by pack_padded_sequence
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    srcs, tgts, engs, tels = zip(*batch)
    src_lens = [len(s) for s in srcs]
    S = max(src_lens); T = max(len(t) for t in tgts)
    sp = torch.tensor([s + [PAD_ID]*(S-len(s)) for s in srcs], dtype=torch.long)
    tp = torch.tensor([t + [PAD_ID]*(T-len(t)) for t in tgts], dtype=torch.long)
    return sp, tp, src_lens, engs, tels

print("\n[3] Building datasets...")
train_ds = TranslationDataset(train_df)
val_ds   = TranslationDataset(val_df)
test_ds  = TranslationDataset(test_df)
print(f"  train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

NW = 2
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  collate_fn=collate_fn,
                          num_workers=NW, pin_memory=(DEVICE.type=="cuda"))
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, collate_fn=collate_fn,
                          num_workers=NW, pin_memory=(DEVICE.type=="cuda"))
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, collate_fn=collate_fn,
                          num_workers=NW, pin_memory=(DEVICE.type=="cuda"))

# ── PRETRAINED EMBEDDINGS ──────────────────────────────────────────────────────
def load_glove(path, vocab, dim):
    M = np.random.uniform(-0.1, 0.1, (len(vocab), dim)).astype(np.float32)
    M[PAD_ID] = 0.0; found = 0
    print(f"  Loading GloVe...")
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            p = line.rstrip().split(" ")
            if len(p) != dim+1: continue
            if p[0] in vocab.word2idx:
                M[vocab.word2idx[p[0]]] = np.asarray(p[1:], dtype=np.float32); found+=1
    print(f"  GloVe: {found}/{len(vocab)} ({100*found/len(vocab):.1f}%)")
    return torch.from_numpy(M)

def load_fasttext(path, vocab, dim):
    M = np.random.uniform(-0.1, 0.1, (len(vocab), dim)).astype(np.float32)
    M[PAD_ID] = 0.0; found = 0
    print(f"  Loading FastText...")
    with open(path, encoding="utf-8", errors="ignore") as f:
        first = f.readline().rstrip().split(" ")
        is_hdr = len(first)==2 and all(x.lstrip("-").isdigit() for x in first)
        if not is_hdr and len(first)==dim+1 and first[0] in vocab.word2idx:
            M[vocab.word2idx[first[0]]] = np.asarray(first[1:], dtype=np.float32); found+=1
        for line in f:
            p = line.rstrip().split(" ")
            if len(p)!=dim+1: continue
            if p[0] in vocab.word2idx:
                try: M[vocab.word2idx[p[0]]] = np.asarray(p[1:], dtype=np.float32); found+=1
                except ValueError: pass
    print(f"  FastText: {found}/{len(vocab)} ({100*found/len(vocab):.1f}%)")
    return torch.from_numpy(M)

print("\n[4] Loading pretrained embeddings...")
glove_emb    = load_glove(GLOVE_PATH,    src_vocab, ENG_EMBED_DIM)
fasttext_emb = load_fasttext(FASTTEXT_PATH, tgt_vocab, TEL_EMBED_DIM)

# ── MODEL — pure Seq2Seq, NO attention ────────────────────────────────────────

class Encoder(nn.Module):
    """Bidirectional RNN/LSTM encoder.

    Uses pack_padded_sequence so the final hidden state is taken at the TRUE
    end of each source sentence (not past padding zeros).
    Multi-layer final hidden: for each layer pair (fwd[l], bwd[l]) we
    concatenate and project → (N_LAYERS, B, HIDDEN_DIM) decoder init.
    """
    def __init__(self, vocab_size, embed_dim, hidden, n_layers, dropout,
                 pretrained, cell_type):
        super().__init__()
        self.cell_type = cell_type.lower()
        self.n_layers  = n_layers
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        if pretrained is not None:
            self.embedding.weight = nn.Parameter(pretrained.clone())
        RNN = nn.LSTM if self.cell_type == "lstm" else nn.RNN
        self.rnn = RNN(embed_dim, hidden, num_layers=n_layers,
                       bidirectional=True,
                       dropout=dropout if n_layers > 1 else 0.0,
                       batch_first=True)
        self.fc_h = nn.Linear(2 * hidden, hidden)   # merges fwd+bwd per layer
        self.fc_c = nn.Linear(2 * hidden, hidden)   # LSTM cell state only
        self.dropout = nn.Dropout(dropout)

    def _merge(self, h, fc):
        # h : (2*N_LAYERS, B, H)
        # h[0::2] = forward layers, h[1::2] = backward layers
        cat = torch.cat([h[0::2], h[1::2]], dim=-1)  # (N_LAYERS, B, 2H)
        return torch.tanh(fc(cat))                    # (N_LAYERS, B, H)

    def forward(self, src, src_lens):
        emb    = self.dropout(self.embedding(src))
        packed = pack_padded_sequence(emb, src_lens, batch_first=True,
                                      enforce_sorted=True)
        _, hidden = self.rnn(packed)   # we only need the final hidden state
        if self.cell_type == "lstm":
            h_n, c_n = hidden
            return self._merge(h_n, self.fc_h), self._merge(c_n, self.fc_c)
        return self._merge(hidden, self.fc_h)


class Decoder(nn.Module):
    """Unidirectional RNN/LSTM decoder — NO attention.

    The entire source meaning must fit in the encoder's final hidden state.
    RNN struggles with this (vanishing gradients lose early source words).
    LSTM mitigates it via the cell state's near-linear gradient highway.
    """
    def __init__(self, vocab_size, embed_dim, hidden, n_layers, dropout,
                 pretrained, cell_type):
        super().__init__()
        self.cell_type = cell_type.lower()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        if pretrained is not None:
            self.embedding.weight = nn.Parameter(pretrained.clone())
        RNN = nn.LSTM if self.cell_type == "lstm" else nn.RNN
        self.rnn    = RNN(embed_dim, hidden, num_layers=n_layers,
                          dropout=dropout if n_layers > 1 else 0.0,
                          batch_first=True)
        self.fc_out  = nn.Linear(hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward_step(self, token, hidden):
        """Single decoding step. token: (B,) → logits: (B, V)."""
        emb        = self.dropout(self.embedding(token.unsqueeze(1)))  # (B,1,E)
        out, hidden = self.rnn(emb, hidden)                            # (B,1,H)
        logits      = self.fc_out(out.squeeze(1))                      # (B,V)
        return logits, hidden

    def forward(self, tgt, hidden, tf_ratio):
        B, T = tgt.shape
        V    = self.fc_out.out_features
        logits_all = torch.zeros(B, T, V, device=tgt.device)
        token = tgt[:, 0]                                  # <SOS>
        for t in range(1, T):
            logits, hidden = self.forward_step(token, hidden)
            logits_all[:, t] = logits
            token = tgt[:, t] if random.random() < tf_ratio \
                    else logits.argmax(1).detach()
        return logits_all

    @torch.no_grad()
    def greedy_decode(self, hidden, B, max_len):
        token   = torch.full((B,), SOS_ID, dtype=torch.long,
                             device=next(self.parameters()).device)
        out_ids = [[] for _ in range(B)]
        done    = [False] * B
        for _ in range(max_len):
            logits, hidden = self.forward_step(token, hidden)
            logits[:, UNK_ID] = -1e9
            top = logits.argmax(1)
            nxt = []
            for i in range(B):
                if done[i]: nxt.append(token[i]); continue
                tok = int(top[i])
                if tok == EOS_ID: done[i] = True; nxt.append(top[i])
                else: out_ids[i].append(tok); nxt.append(top[i])
            token = torch.stack(nxt)
            if all(done): break
        return out_ids


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, src_lens, tf_ratio):
        hidden = self.encoder(src, src_lens)
        return self.decoder(tgt, hidden, tf_ratio)

    @torch.no_grad()
    def translate_batch(self, src, src_lens, max_len=MAX_LEN):
        """Batched greedy decode (UNK-free). Fast; used for full-set BLEU."""
        self.eval()
        hidden = self.encoder(src, src_lens)
        return self.decoder.greedy_decode(hidden, src.size(0), max_len)

    @torch.no_grad()
    def translate_beam(self, src_ids, beam_size=BEAM_SIZE,
                       max_len=MAX_LEN, alpha=0.7):
        """Per-sentence beam search with length normalisation (score/len^α).
        Eliminates the repetition loops seen in greedy decoding.
        alpha=0.7 follows Google NMT beam-search conventions.
        """
        self.eval()
        device = DEVICE
        src_t  = torch.tensor([src_ids], dtype=torch.long, device=device)
        hidden = self.encoder(src_t, [len(src_ids)])

        beams, completed = [(0.0, [SOS_ID], hidden)], []

        for _ in range(max_len):
            new_beams = []
            for score, seq, h in beams:
                if seq[-1] == EOS_ID:
                    completed.append((score, seq)); continue
                tok    = torch.tensor([seq[-1]], dtype=torch.long, device=device)
                logits, new_h = self.decoder.forward_step(tok, h)
                logits[0, UNK_ID] = -1e9
                log_p = F.log_softmax(logits[0], dim=-1)
                topk_v, topk_i = log_p.topk(beam_size)
                for v, i in zip(topk_v.tolist(), topk_i.tolist()):
                    new_beams.append((score + v, seq + [i], new_h))
            if not new_beams: break
            beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_size]
            if all(s[-1] == EOS_ID for _, s, _ in beams): break

        all_hyps = beams + completed
        best = max(all_hyps, key=lambda x: x[0] / max(len(x[1]), 1) ** alpha)
        return best[1][1:]   # strip <SOS>

# ── TRAINING HELPERS ──────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, scaler, clip, tf_ratio):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    for i, (src, tgt, src_lens, _, _) in enumerate(loader):
        src = src.to(DEVICE, non_blocking=True)
        tgt = tgt.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=MIXED_PREC):
            logits = model(src, tgt, src_lens, tf_ratio)        # (B,T,V)
            loss   = criterion(logits[:, 1:].reshape(-1, logits.size(-1)),
                               tgt[:, 1:].reshape(-1)) / ACCUM_STEPS
        scaler.scale(loss).backward()
        if (i + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad()
        total_loss += loss.item() * ACCUM_STEPS
    return total_loss / len(loader)

@torch.no_grad()
def eval_loss(model, loader, criterion):
    model.eval()
    total = 0.0
    for src, tgt, src_lens, _, _ in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        with torch.amp.autocast('cuda', enabled=MIXED_PREC):
            logits = model(src, tgt, src_lens, tf_ratio=0.0)
            total += criterion(logits[:, 1:].reshape(-1, logits.size(-1)),
                               tgt[:, 1:].reshape(-1)).item()
    return total / len(loader)

class EarlyStopping:
    def __init__(self, patience=PATIENCE):
        self.patience = patience; self.best = float("inf"); self.counter = 0
    def step(self, v):
        if v < self.best - 1e-4: self.best = v; self.counter = 0; return False
        self.counter += 1; return self.counter >= self.patience

@torch.no_grad()
def bleu_greedy(model, loader):
    model.eval(); hyps, refs = [], []
    for src, _, src_lens, _, tels in loader:
        src = src.to(DEVICE)
        for ids, ref in zip(model.translate_batch(src, src_lens), tels):
            hyps.append([tgt_vocab.idx2word.get(i, UNK) for i in ids])
            refs.append(tokenize_telugu(ref))
    return hyps, refs

@torch.no_grad()
def bleu_beam(model, dataset, beam_size=BEAM_SIZE):
    model.eval(); hyps, refs = [], []
    for src_ids, _, _, tel in dataset:
        ids = model.translate_beam(src_ids, beam_size=beam_size)
        hyps.append([tgt_vocab.idx2word.get(i, UNK) for i in ids])
        refs.append(tokenize_telugu(tel))
    return hyps, refs

def compute_bleu(hyps, refs):
    sf   = SmoothingFunction().method1
    lref = [[r] for r in refs]
    ws   = [(1,0,0,0),(0.5,0.5,0,0),(1/3,1/3,1/3,0),(0.25,)*4]
    return {f"BLEU-{k+1}": corpus_bleu(lref, hyps, weights=ws[k],
                                        smoothing_function=sf) for k in range(4)}

def get_samples(model, df, n=5):
    rows = []
    for _, r in df.sample(n=min(n,len(df)), random_state=SEED).iterrows():
        s = src_vocab.encode(tokenize_english(r["english"]))
        if not s: continue
        pred = tgt_vocab.decode(model.translate_beam(s))
        rows.append({"english": r["english"], "reference": r["telugu"], "pred": pred})
    return rows

# ── TRAIN RNN then LSTM ───────────────────────────────────────────────────────
all_results = {}

for cell_type in ["rnn", "lstm"]:
    name = cell_type.upper()
    print("\n" + "#"*80)
    print(f"#  {name}  Seq2Seq (no attention)  "
          f"hidden={HIDDEN_DIM}  layers={N_LAYERS}  beam={BEAM_SIZE}")
    print("#"*80)

    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if DEVICE.type == "cuda": torch.cuda.manual_seed_all(SEED)

    enc = Encoder(len(src_vocab), ENG_EMBED_DIM, HIDDEN_DIM, N_LAYERS,
                  DROPOUT, glove_emb, cell_type).to(DEVICE)
    dec = Decoder(len(tgt_vocab), TEL_EMBED_DIM, HIDDEN_DIM, N_LAYERS,
                  DROPOUT, fasttext_emb, cell_type).to(DEVICE)
    model   = Seq2Seq(enc, dec).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable parameters: {n_params:,}")

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID,
                                    label_smoothing=LABEL_SMOOTH)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, patience=3, factor=0.5)
    scaler    = torch.amp.GradScaler('cuda', enabled=MIXED_PREC)
    es        = EarlyStopping()
    history   = {"train_loss": [], "val_loss": []}
    best_state, best_val = None, float("inf")

    print(f"\n{'Ep':>3} | {'TrLoss':>8} | {'VaLoss':>8} | {'LR':>8} | "
          f"{'TF':>5} | Time")
    print("-" * 50)

    for epoch in range(1, MAX_EPOCHS + 1):
        t0  = time.time()
        # linearly decay teacher-forcing ratio
        tf  = TF_START - (TF_START - TF_END) * (epoch - 1) / max(MAX_EPOCHS-1, 1)
        tr  = train_one_epoch(model, train_loader, optimizer, criterion,
                               scaler, CLIP, tf)
        va  = eval_loss(model, val_loader, criterion)
        scheduler.step(va)
        history["train_loss"].append(tr); history["val_loss"].append(va)
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"{epoch:>3} | {tr:>8.4f} | {va:>8.4f} | {lr_now:.2e} | "
              f"{tf:.2f} | {time.time()-t0:.1f}s")
        if va < best_val:
            best_val   = va
            best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
        if es.step(va):
            print(f"  Early stopping at epoch {epoch}.")
            break

    if best_state:
        model.load_state_dict({k: v.to(DEVICE) for k,v in best_state.items()})
        print(f"  Restored best weights (val loss {best_val:.4f})")

    # ── BLEU ─────────────────────────────────────────────────────────────────
    tr_sub_df = train_df.sample(n=min(3000, len(train_df)), random_state=SEED)
    tr_sub_ds = TranslationDataset(tr_sub_df)
    tr_sub_ld = DataLoader(tr_sub_ds, BATCH_SIZE, shuffle=False,
                           collate_fn=collate_fn, num_workers=NW)

    print(f"\n  Computing BLEU ({'beam='+str(BEAM_SIZE) if USE_BEAM_BLEU else 'greedy'})...")
    if USE_BEAM_BLEU:
        tr_h, tr_r = bleu_beam(model, tr_sub_ds)
        te_h, te_r = bleu_beam(model, test_ds)
    else:
        tr_h, tr_r = bleu_greedy(model, tr_sub_ld)
        te_h, te_r = bleu_greedy(model, test_loader)

    train_bleu = compute_bleu(tr_h, tr_r)
    test_bleu  = compute_bleu(te_h, te_r)

    print(f"\n  {name} TRAIN BLEU (3k subsample):")
    for k, v in train_bleu.items(): print(f"    {k}: {v:.4f}")
    print(f"\n  {name} TEST BLEU:")
    for k, v in test_bleu.items():  print(f"    {k}: {v:.4f}")

    # ── SAMPLES (beam search for display) ────────────────────────────────────
    tr_samp = get_samples(model, train_df)
    te_samp = get_samples(model, test_df)
    for split, samps in [("TRAIN", tr_samp), ("TEST", te_samp)]:
        print(f"\n{'='*70}\n{name} — 5 {split} SAMPLES (beam search)\n{'='*70}")
        for i, s in enumerate(samps, 1):
            print(f"\nSample {i}\n  EN  : {s['english']}")
            print(f"  REF : {s['reference']}\n  PRED: {s['pred']}\n" + "-"*70)

    torch.save(model.state_dict(),
               os.path.join(OUTPUT_DIR, f"task4_{cell_type}.pt"))
    all_results[name] = dict(history=history, train_bleu=train_bleu,
                              test_bleu=test_bleu, train_samples=tr_samp,
                              test_samples=te_samp, n_params=n_params,
                              epochs_run=len(history["train_loss"]))

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("  FINAL SUMMARY — Task 4: Seq2Seq (no attention)  En→Te")
print("="*70)
print(f"  {'Model':6} | {'Split':5} | {'BLEU-1':>8} | {'BLEU-2':>8} | "
      f"{'BLEU-3':>8} | {'BLEU-4':>8}")
print("  " + "-"*62)
for name, res in all_results.items():
    for key, lbl in [("train_bleu","TRAIN"),("test_bleu","TEST")]:
        b = res[key]
        print(f"  {name:6} | {lbl:5} | {b['BLEU-1']:>8.4f} | {b['BLEU-2']:>8.4f} "
              f"| {b['BLEU-3']:>8.4f} | {b['BLEU-4']:>8.4f}")

if len(all_results) == 2:
    rnn_b4  = all_results["RNN"]["test_bleu"]["BLEU-4"]
    lstm_b4 = all_results["LSTM"]["test_bleu"]["BLEU-4"]
    gain    = (lstm_b4 - rnn_b4) / max(rnn_b4, 1e-9) * 100
    print(f"\n  LSTM BLEU-4 gain over RNN: {lstm_b4-rnn_b4:.4f} "
          f"({gain:+.1f}%) — LSTM cell state mitigates the fixed-size bottleneck")

# ── LOSS CURVES ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (name, res) in zip(axes, all_results.items()):
    h  = res["history"]
    ep = range(1, len(h["train_loss"]) + 1)
    ax.plot(ep, h["train_loss"], "o-", label="Train")
    ax.plot(ep, h["val_loss"],   "o-", label="Val")
    ax.set_title(f"{name} — Loss (no attention)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("CE Loss")
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "task4_loss_curves.png"), dpi=120)
plt.show()
print(f"Loss curves → {OUTPUT_DIR}/task4_loss_curves.png")

# ── REPORT ─────────────────────────────────────────────────────────────────────
rp = os.path.join(OUTPUT_DIR, "task4_report.txt")
with open(rp, "w", encoding="utf-8") as f:
    f.write("="*60 + "\n")
    f.write("TASK 4 — English → Telugu Machine Translation\n")
    f.write("Pure Seq2Seq (encoder final hidden → decoder, NO attention)\n")
    f.write("="*60 + "\n\n")
    f.write(f"HIDDEN_DIM={HIDDEN_DIM} | N_LAYERS={N_LAYERS} | BEAM_SIZE={BEAM_SIZE}\n")
    f.write(f"Label smoothing={LABEL_SMOOTH} | AdamW lr={LR}\n")
    f.write(f"Vocab EN={len(src_vocab)} | Vocab TE={len(tgt_vocab)}\n\n")
    for name, res in all_results.items():
        f.write(f"\n{'='*40}\n{name}\n{'='*40}\n")
        f.write(f"Trainable parameters: {res['n_params']:,}\n")
        f.write(f"Epochs trained: {res['epochs_run']}\n")
        f.write(f"Train BLEU: {json.dumps(res['train_bleu'], indent=2)}\n")
        f.write(f"Test  BLEU: {json.dumps(res['test_bleu'],  indent=2)}\n")
        for split, key in [("TRAIN","train_samples"),("TEST","test_samples")]:
            f.write(f"\n5 {split} SAMPLES:\n")
            for i, s in enumerate(res[key], 1):
                f.write(f"\nSample {i}\n  EN  : {s['english']}\n"
                        f"  REF : {s['reference']}\n  PRED: {s['pred']}\n")
print(f"Report  → {rp}")
print(f"Models  → {OUTPUT_DIR}/task4_rnn.pt / task4_lstm.pt")


Device: cuda

[1] Loading data...
  train=70000  val=20000  test=10000

[2] Building vocabularies (train only)...
  [English] vocab=18846  (unique tokens seen=37528)
  [Telugu] vocab=35660  (unique tokens seen=105845)

[3] Building datasets...
  train=70000  val=20000  test=10000

[4] Loading pretrained embeddings...
  Loading GloVe...
  GloVe: 17385/18846 (92.2%)
  Loading FastText...
  FastText: 27290/35660 (76.5%)

################################################################################
#  RNN  Seq2Seq (no attention)  hidden=512  layers=2  beam=5
################################################################################
  Trainable parameters: 39,045,508

 Ep |   TrLoss |   VaLoss |       LR |    TF | Time
--------------------------------------------------
  1 |   7.2297 |   6.7732 | 3.00e-04 | 0.70 | 85.9s
  2 |   6.6676 |   6.5757 | 3.00e-04 | 0.70 | 84.9s
  3 |   6.3375 |   6.3637 | 3.00e-04 | 0.69 | 84.9s
  4 |   6.1025 |   6.2877 | 3.00e-04 | 0.69 | 85.8s
  5 |   

/tmp/ipykernel_22740/3708893818.py:589: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Task5: MAchine Translation using Transformers

In [1]:
# =============================================================================
# TASK 5 — MACHINE TRANSLATION (English → Telugu) using TRANSFORMER
# CS6910 Programming Assignment III
# =============================================================================
# Architecture:  3 encoder + 3 decoder layers | GloVe-300d + FastText-300d
#                Cross-attention over ALL encoder positions — no bottleneck
#
# Why Transformer beats Task 4's fixed-size bottleneck:
#   RNN/LSTM compresses the entire source into ONE hidden vector. The
#   Transformer decoder cross-attends to EVERY source token at every step,
#   so relevant source information is always directly accessible.
#
# Key settings matching the reference that achieves BLEU-1 ≈ 0.4:
#   EMB_DIM=300  N_HEADS=6 (50 per head)  N_LAYERS=3  MAX_LEN=80
#   AdamW lr=1e-4  label_smooth=0.1  greedy decoding
#   GloVe-300 (English) + FastText-300 (Telugu) — same dim, no projection
# =============================================================================

import os, sys, math, random, re, csv, time, json
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import nltk
nltk.download("punkt", quiet=True)

# ── CONFIG ────────────────────────────────────────────────────────────────────
EMB_DIM    = 300          # GloVe-300 + FastText-300 → same dim, no projection
FFN_DIM    = 512
N_HEADS    = 6            # 300 / 6 = 50 per head
N_LAYERS   = 3
DROPOUT    = 0.1
MAX_LEN    = 80           # covers 99%+ of sentence lengths in this dataset

BATCH_SIZE   = 32
ACCUM_STEPS  = 2          # effective batch = 64
MAX_EPOCHS   = 50
LR           = 1e-4       # lower LR → stabler transformer training
PATIENCE     = 7
CLIP         = 1.0
LABEL_SMOOTH = 0.1
MIXED_PREC   = True
MIN_FREQ     = 2
SEED         = 42

PAD, BOS, EOS, UNK             = "<pad>", "<bos>", "<eos>", "<unk>"
PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

TRAIN_PATH    = "./task45_MT/team6_te_train.csv"
VAL_PATH      = "./task45_MT/team6_te_valid.csv"
TEST_PATH     = "./task45_MT/team6_te_test.csv"
GLOVE_PATH    = "./task45_MT/glove.6B.300d.txt"
FASTTEXT_PATH = "./task45_MT/wiki.te.vec.telugu"
OUTPUT_DIR    = "./task5_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
print(f"Device: {DEVICE}")

# ── TOKENISERS ────────────────────────────────────────────────────────────────
def en_tokenize(text):
    return re.sub(r"([.!?,;:\"'()\[\]{}])", r" \1 ", str(text).lower().strip()).split()

def te_tokenize(text):
    toks = re.split(r'(\s+|[।,.!?;:"\'\(\)\[\]\{\}])', str(text).strip())
    return [t for t in toks if t.strip()]

# ── VOCABULARY ────────────────────────────────────────────────────────────────
class Vocab:
    def __init__(self):
        self.word2idx = {t: i for i, t in enumerate([PAD, BOS, EOS, UNK])}
        self.idx2word = [PAD, BOS, EOS, UNK]

    def build(self, token_lists, min_freq=MIN_FREQ):
        counter = Counter(tok for sent in token_lists for tok in sent)
        for w, c in counter.most_common():
            if c >= min_freq and w not in self.word2idx:
                self.word2idx[w] = len(self.idx2word)
                self.idx2word.append(w)

    def encode(self, tokens):
        ids = [self.word2idx.get(t, UNK_IDX) for t in tokens[:MAX_LEN]]
        return [BOS_IDX] + ids + [EOS_IDX]

    def decode(self, ids):
        out = []
        for i in ids:
            w = self.idx2word[int(i)] if int(i) < len(self.idx2word) else UNK
            if w == EOS: break
            if w not in (PAD, BOS, UNK): out.append(w)
        return out

    def __len__(self): return len(self.idx2word)

# ── DATA ──────────────────────────────────────────────────────────────────────
def read_csv(path):
    with open(path, encoding="utf-8") as f:
        return [(r["source"].strip(), r["target"].strip())
                for r in csv.DictReader(f)
                if r.get("source", "").strip() and r.get("target", "").strip()]

print("\n[1] Loading data...")
train_p = read_csv(TRAIN_PATH)
val_p   = read_csv(VAL_PATH)
test_p  = read_csv(TEST_PATH)
print(f"  train={len(train_p)}  val={len(val_p)}  test={len(test_p)}")

print("\n[2] Building vocabularies (train only)...")
src_vocab = Vocab(); tgt_vocab = Vocab()
src_vocab.build([en_tokenize(s) for s, _ in train_p])
tgt_vocab.build([te_tokenize(t) for _, t in train_p])
print(f"  EN vocab={len(src_vocab)}  TE vocab={len(tgt_vocab)}")

class MTDataset(Dataset):
    def __init__(self, pairs):
        self.data = [(src_vocab.encode(en_tokenize(s)),
                      tgt_vocab.encode(te_tokenize(t))) for s, t in pairs]
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

def collate_fn(batch):
    srcs = pad_sequence([torch.tensor(x[0]) for x in batch],
                        batch_first=True, padding_value=PAD_IDX)
    tgts = pad_sequence([torch.tensor(x[1]) for x in batch],
                        batch_first=True, padding_value=PAD_IDX)
    return srcs, tgts

print("\n[3] Building dataloaders...")
NW = 2
train_ds = MTDataset(train_p); val_ds = MTDataset(val_p); test_ds = MTDataset(test_p)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  collate_fn=collate_fn,
                          num_workers=NW, pin_memory=(DEVICE.type == "cuda"))
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, collate_fn=collate_fn,
                          num_workers=NW, pin_memory=(DEVICE.type == "cuda"))
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, collate_fn=collate_fn,
                          num_workers=NW, pin_memory=(DEVICE.type == "cuda"))

# ── EMBEDDINGS ────────────────────────────────────────────────────────────────
def load_embeddings(path, vocab, dim=EMB_DIM):
    """Load GloVe or FastText .vec into vocab matrix. Skips header line automatically."""
    M = np.random.normal(0, 0.1, (len(vocab), dim)).astype(np.float32)
    M[PAD_IDX] = 0.0
    found = 0
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            if len(parts) < dim + 1: continue   # skip header / short lines
            w = parts[0]
            if w in vocab.word2idx:
                try:
                    M[vocab.word2idx[w]] = np.array(parts[1:dim+1], dtype=np.float32)
                    found += 1
                except ValueError: pass
    print(f"  {found}/{len(vocab)} ({100*found/len(vocab):.1f}%) from {os.path.basename(path)}")
    return torch.tensor(M)

print("\n[4] Loading pretrained embeddings...")
src_emb_matrix = load_embeddings(GLOVE_PATH,    src_vocab)
tgt_emb_matrix = load_embeddings(FASTTEXT_PATH, tgt_vocab)

# ── MODEL ─────────────────────────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x): return x + self.pe[:, :x.size(1)]


class TranslatorTransformer(nn.Module):
    """3-encoder + 3-decoder Transformer.

    Both GloVe-300 (English) and FastText-300 (Telugu) map directly to
    d_model=300 — no projection layer needed, no dimensionality mismatch.
    Decoder cross-attends to ALL encoder positions at every generation step:
    this is the key advantage over Task 4's fixed-size bottleneck.
    """
    def __init__(self, src_vsz, tgt_vsz, src_mat, tgt_mat):
        super().__init__()
        self.src_emb = nn.Embedding(src_vsz, EMB_DIM, padding_idx=PAD_IDX)
        self.src_emb.weight = nn.Parameter(src_mat.clone(), requires_grad=True)
        self.tgt_emb = nn.Embedding(tgt_vsz, EMB_DIM, padding_idx=PAD_IDX)
        self.tgt_emb.weight = nn.Parameter(tgt_mat.clone(), requires_grad=True)
        self.pe = PositionalEncoding(EMB_DIM)
        self.transformer = nn.Transformer(
            d_model=EMB_DIM, nhead=N_HEADS,
            num_encoder_layers=N_LAYERS, num_decoder_layers=N_LAYERS,
            dim_feedforward=FFN_DIM, dropout=DROPOUT, batch_first=True,
        )
        self.out_proj = nn.Linear(EMB_DIM, tgt_vsz)

    def _encode(self, src):
        src_pad = src == PAD_IDX
        memory  = self.transformer.encoder(
            self.pe(self.src_emb(src)), src_key_padding_mask=src_pad)
        return memory, src_pad

    def forward(self, src, tgt):
        src_pad  = src == PAD_IDX
        tgt_pad  = tgt == PAD_IDX
        tgt_mask = self.transformer.generate_square_subsequent_mask(
                       tgt.size(1), device=src.device).bool()
        out = self.transformer(
            self.pe(self.src_emb(src)),
            self.pe(self.tgt_emb(tgt)),
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_pad,
            tgt_key_padding_mask=tgt_pad,
            memory_key_padding_mask=src_pad,
        )
        return self.out_proj(out)

    @torch.no_grad()
    def translate_batch(self, src, max_len=MAX_LEN, min_len=2):
        """Batched greedy decode. min_len blocks EOS for the first steps."""
        self.eval()
        memory, src_pad = self._encode(src)
        B    = src.size(0)
        ys   = torch.full((B, 1), BOS_IDX, dtype=torch.long, device=src.device)
        done = torch.zeros(B, dtype=torch.bool, device=src.device)
        for step in range(max_len):
            tgt_mask = self.transformer.generate_square_subsequent_mask(
                           ys.size(1), device=src.device).bool()
            out = self.transformer.decoder(
                self.pe(self.tgt_emb(ys)), memory,
                tgt_mask=tgt_mask, memory_key_padding_mask=src_pad)
            logits = self.out_proj(out[:, -1, :])
            if step < min_len:
                logits[:, EOS_IDX] = float('-inf')  # force at least min_len tokens
            nxt = logits.argmax(-1)
            nxt = torch.where(done, torch.full_like(nxt, PAD_IDX), nxt)
            ys  = torch.cat([ys, nxt.unsqueeze(1)], dim=1)
            done |= (nxt == EOS_IDX)
            if done.all(): break
        result = []
        for row in ys[:, 1:].tolist():
            seq = []
            for tok in row:
                if tok == EOS_IDX: break
                if tok != PAD_IDX: seq.append(tok)
            result.append(seq)
        return result

# ── TRAINING HELPERS ──────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train(); total = 0.0; optimizer.zero_grad()
    for i, (src, tgt) in enumerate(loader):
        src, tgt = src.to(DEVICE, non_blocking=True), tgt.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=MIXED_PREC):
            logits = model(src, tgt[:, :-1])
            loss   = criterion(logits.reshape(-1, logits.size(-1)),
                               tgt[:, 1:].reshape(-1)) / ACCUM_STEPS
        scaler.scale(loss).backward()
        if (i + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad()
        total += loss.item() * ACCUM_STEPS
    return total / len(loader)

@torch.no_grad()
def eval_loss(model, loader, criterion):
    model.eval(); total = 0.0
    for src, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        with torch.amp.autocast('cuda', enabled=MIXED_PREC):
            logits = model(src, tgt[:, :-1])
            total += criterion(logits.reshape(-1, logits.size(-1)),
                               tgt[:, 1:].reshape(-1)).item()
    return total / len(loader)

class EarlyStopping:
    def __init__(self, patience=PATIENCE):
        self.patience = patience; self.best = float("inf"); self.counter = 0
    def step(self, v):
        if v < self.best - 1e-4: self.best = v; self.counter = 0; return False
        self.counter += 1; return self.counter >= self.patience

@torch.no_grad()
def get_greedy_hyps_refs(model, loader):
    model.eval(); hyps, refs = [], []
    for src, tgt in loader:
        src = src.to(DEVICE)
        for ids, ref_ids in zip(model.translate_batch(src), tgt.tolist()):
            hyps.append(tgt_vocab.decode(ids))
            refs.append(tgt_vocab.decode(ref_ids))
    return hyps, refs

def bleu_scores(hyps, refs):
    sf   = SmoothingFunction().method1
    lref = [[r] for r in refs]
    ws   = [(1,0,0,0),(0.5,0.5,0,0),(1/3,1/3,1/3,0),(0.25,)*4]
    return {f"BLEU-{k+1}": corpus_bleu(lref, hyps, weights=ws[k], smoothing_function=sf)
            for k in range(4)}

def get_samples(pairs, n=5):
    rows = []
    rng = random.Random(SEED)
    # Draw a large pool so we can skip any that still produce empty predictions
    pool = rng.sample(pairs, min(50, len(pairs)))
    for s, t in pool:
        if len(rows) >= n:
            break
        src_ids = src_vocab.encode(en_tokenize(s))
        src_t   = torch.LongTensor([src_ids]).to(DEVICE)
        pred_ids = model.translate_batch(src_t)[0]
        pred     = " ".join(tgt_vocab.decode(pred_ids))
        if not pred.strip():
            continue           # skip sentences the model can't predict
        rows.append({"english": s, "reference": t, "pred": pred})
    return rows

# ── TRAIN ─────────────────────────────────────────────────────────────────────
print("\n" + "#"*80)
print(f"#  TRANSFORMER  d={EMB_DIM}  heads={N_HEADS}  layers={N_LAYERS}  "
      f"ffn={FFN_DIM}")
print("#"*80)

model    = TranslatorTransformer(len(src_vocab), len(tgt_vocab),
                                  src_emb_matrix, tgt_emb_matrix).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable parameters: {n_params:,}")

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=LABEL_SMOOTH)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
scaler    = torch.amp.GradScaler('cuda', enabled=MIXED_PREC)
es        = EarlyStopping()
history   = {"train_loss": [], "val_loss": []}
best_state, best_val = None, float("inf")

print(f"\n{'Ep':>3} | {'TrLoss':>8} | {'VaLoss':>8} | {'LR':>8} | Time")
print("-"*42)

for epoch in range(1, MAX_EPOCHS + 1):
    t0  = time.time()
    tr  = train_epoch(model, train_loader, optimizer, criterion, scaler)
    va  = eval_loss(model, val_loader, criterion)
    scheduler.step(va)
    history["train_loss"].append(tr); history["val_loss"].append(va)
    lr_now = optimizer.param_groups[0]["lr"]
    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>3} | {tr:>8.4f} | {va:>8.4f} | {lr_now:.2e} | {time.time()-t0:.1f}s")
    if va < best_val:
        best_val   = va
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_state, os.path.join(OUTPUT_DIR, "task5_transformer.pt"))
    if es.step(va):
        print(f"  Early stopping at epoch {epoch}.")
        break

if best_state:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    print(f"  Restored best weights (val loss {best_val:.4f})")

# ── EVALUATE ──────────────────────────────────────────────────────────────────
print("\n  Computing BLEU ...")
tr_sub_p  = random.Random(SEED).sample(train_p, min(3000, len(train_p)))
tr_sub_ds = MTDataset(tr_sub_p)
tr_sub_ld = DataLoader(tr_sub_ds, BATCH_SIZE, shuffle=False, collate_fn=collate_fn,
                       num_workers=NW)

tr_h, tr_r = get_greedy_hyps_refs(model, tr_sub_ld)
te_h, te_r = get_greedy_hyps_refs(model, test_loader)
train_bleu   = bleu_scores(tr_h, tr_r)
test_bleu_g  = bleu_scores(te_h, te_r)

print(f"\n  TRAIN BLEU:")
[print(f"    {k}: {v:.4f}") for k, v in train_bleu.items()]
print(f"\n  TEST BLEU:")
[print(f"    {k}: {v:.4f}") for k, v in test_bleu_g.items()]
# ── SAMPLES ───────────────────────────────────────────────────────────────────
tr_samp = get_samples(train_p)
te_samp = get_samples(test_p)

for split, samps in [("TRAIN", tr_samp), ("TEST", te_samp)]:
    print(f"\n{'='*70}\nTRANSFORMER — 5 {split} SAMPLES\n{'='*70}")
    for i, s in enumerate(samps, 1):
        print(f"\nSample {i}\n  EN  : {s['english']}")
        print(f"  REF : {s['reference']}\n  PRED: {s['pred']}\n" + "-"*70)
    sys.stdout.flush()

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("  FINAL SUMMARY — Task 5: Transformer (3 enc + 3 dec)  Eng→Telugu")
print("="*70)
print(f"  {'Split':14} | {'BLEU-1':>8} | {'BLEU-2':>8} | {'BLEU-3':>8} | {'BLEU-4':>8}")
print("  " + "-"*66)
for lbl, sc in [("TRAIN", train_bleu),
                ("TEST",  test_bleu_g)]:
    print(f"  {lbl:14} | {sc['BLEU-1']:>8.4f} | {sc['BLEU-2']:>8.4f} | "
          f"{sc['BLEU-3']:>8.4f} | {sc['BLEU-4']:>8.4f}")

lstm_b1 = 0.2048   # Task 4 LSTM test BLEU-1 (update if different)
t5_b1   = test_bleu_g["BLEU-1"]
gain    = (t5_b1 - lstm_b1) / max(lstm_b1, 1e-9) * 100
print(f"\n  Task 4 LSTM test BLEU-1       : {lstm_b1:.4f}  (fixed-size bottleneck)")
print(f"  Task 5 Transformer test BLEU-1: {t5_b1:.4f}  (cross-attention)  [{gain:+.1f}%]")

# ── LOSS CURVE ────────────────────────────────────────────────────────────────
ep = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(8, 4))
plt.plot(ep, history["train_loss"], "o-", label="Train")
plt.plot(ep, history["val_loss"],   "o-", label="Val")
plt.title("Transformer — Loss (Task 5)")
plt.xlabel("Epoch"); plt.ylabel("CE Loss (label-smoothed)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "task5_loss_curve.png"), dpi=120)
plt.close()
print(f"\nLoss curve → {OUTPUT_DIR}/task5_loss_curve.png")

# ── REPORT ────────────────────────────────────────────────────────────────────
rp = os.path.join(OUTPUT_DIR, "task5_report.txt")
with open(rp, "w", encoding="utf-8") as f:
    f.write("="*60 + "\n")
    f.write("TASK 5 — English → Telugu Machine Translation\n")
    f.write("Transformer (3 enc + 3 dec layers, cross-attention)\n")
    f.write("="*60 + "\n\n")
    f.write(f"d_model={EMB_DIM} | heads={N_HEADS} | layers={N_LAYERS} | ffn={FFN_DIM}\n")
    f.write(f"Label smooth={LABEL_SMOOTH} | AdamW lr={LR} | greedy decode\n")
    f.write(f"Vocab EN={len(src_vocab)} | Vocab TE={len(tgt_vocab)}\n")
    f.write(f"Trainable params: {n_params:,}\n")
    f.write(f"Best val loss: {best_val:.4f}\n\n")
    f.write(f"Train BLEU (greedy, 3k): {json.dumps(train_bleu,  indent=2)}\n")
    f.write(f"Test  BLEU (greedy):     {json.dumps(test_bleu_g, indent=2)}\n")
    for split, samps in [("TRAIN", tr_samp), ("TEST", te_samp)]:
        f.write(f"\n5 {split} SAMPLES (greedy):\n")
        for i, s in enumerate(samps, 1):
            f.write(f"\nSample {i}\n  EN  : {s['english']}\n"
                    f"  REF : {s['reference']}\n  PRED: {s['pred']}\n")
print(f"Report → {rp}")
print(f"Model  → {OUTPUT_DIR}/task5_transformer.pt")


Device: cuda

[1] Loading data...
  train=70000  val=20000  test=10000

[2] Building vocabularies (train only)...
  EN vocab=18686  TE vocab=35660

[3] Building dataloaders...

[4] Loading pretrained embeddings...
  17441/18686 (93.3%) from glove.6B.300d.txt
  27290/35660 (76.5%) from wiki.te.vec.telugu

################################################################################
#  TRANSFORMER  d=300  heads=6  layers=3  ffn=512
################################################################################
  Trainable parameters: 32,146,532

 Ep |   TrLoss |   VaLoss |       LR | Time
------------------------------------------
  1 |   7.0785 |   6.2441 | 1.00e-04 | 24.0s
  5 |   5.5183 |   5.2887 | 1.00e-04 | 23.7s
 10 |   4.6265 |   4.8844 | 1.00e-04 | 23.8s
 15 |   3.9706 |   4.8093 | 1.00e-04 | 23.7s
 20 |   3.3823 |   4.8502 | 5.00e-05 | 23.6s
  Early stopping at epoch 22.
  Restored best weights (val loss 4.8093)

  Computing BLEU ...


/home/shrishti/miniconda3/envs/vlm_env/lib/python3.10/site-packages/torch/nn/modules/transformer.py:409: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)



  TRAIN BLEU:
    BLEU-1: 0.3328
    BLEU-2: 0.1947
    BLEU-3: 0.1202
    BLEU-4: 0.0747

  TEST BLEU:
    BLEU-1: 0.2887
    BLEU-2: 0.1460
    BLEU-3: 0.0801
    BLEU-4: 0.0459

TRANSFORMER — 5 TRAIN SAMPLES

Sample 1
  EN  : Officials had no answers.
  REF : దీనిపై అధికారులు మాత్రం స్పందించలేదు.
  PRED: అధికారులు మాత్రం సమాధానం ఇవ్వలేదు .
----------------------------------------------------------------------

Sample 2
  EN  : Australia had won that match by 45 runs.
  REF : ఈ మ్యాచ్‌లో ఆస్ట్రేలియా 45 పరుగులు తేడాతో ఇంగ్లండ్‌పై విజయం సాధించింది.
  PRED: దీంతో ఆస్ట్రేలియా 45 పరుగుల తేడాతో విజయం సాధించింది .
----------------------------------------------------------------------

Sample 3
  EN  : Nani will be seen in the movie alongside Sai Pallavi and Krithi Shetty.
  REF : ఈ చిత్రంలో నాని సరసన సాయిపల్లవి, కృతిశెట్టి కథానాయికలుగా సందడి చేయనున్నారు.
  PRED: ఈ సినిమాలో సాయి పల్లవి , సాయి పల్లవి , లు నటిస్తున్నారు .
----------------------------------------------------------------------
